In [7]:
import requests
import pandas as pd
import time

In [8]:
ACCESS_TOKEN = "vk1.a.dn7C3C24UdPqRzs7Z5daRM4LgGfCqQkVzuFcur8_cdMYCfnlnCXSJTg1y3MLmU7r4ycgG8KbfAiDaR1eHT4UjnMuwMdjS9NBmsDb_ky2LPjTOQ8MLEpUzZuML-qrGvJnMLZwT832vmhSMWIUsnbGGdT6Wxh1NBp7aOXUW1EIHHMYoP74erzHSVIlJ3IK13R8-jN4J-kF7121heIJfiv_nA"

In [9]:
GROUP_IDS = {
    "Женский форум": "-211229778",
    "LABELCOM": "-209976560",
    "Импроком": "-203677279",
    "ОХ": "-165221845",
    "Азамат Мусугалиев": "-110135406",
    "ТОП": "-159848117",
    "ВПИСКА": "-149430811"
}
VERSION = "5.199"

BASE_URL = "https://api.vk.com/method/"

def vk_request(method, params):
    url = f"{BASE_URL}{method}"
    params.update({"access_token": ACCESS_TOKEN, "v": VERSION})
    response = requests.get(url, params=params).json()
    if "error" in response:
        print(f"Ошибка: {response['error']['error_msg']}")
        return None
    return response.get("response", {})

def get_videos(group_id):
    count = 200
    offset = 0
    videos = []
    while True:
        params = {
            "owner_id": group_id,
            "count": count,
            "offset": offset,
            "extended": 1,
            "fields": "groups",
        }
        data = vk_request("video.get", params)
        items = data["items"]
        videos.extend(items)
        offset += count
        if offset >= data.get("count", 0):
            break
        time.sleep(0.3)
    return videos


all_videos = []
for name, group in GROUP_IDS.items():
    print(f"Видео из сообщества {group}...")
    videos = get_videos(group)
    print(f"Найдено видео: {len(videos)}")
    all_videos.extend(videos)
    time.sleep(0.5)

df = pd.DataFrame(all_videos)

if "likes" in df.columns:
    df["likes_count"] = df["likes"].apply(lambda x: x.get("count")
    if isinstance(x, dict) else 0)
if "reposts" in df.columns:
    df["reposts_count"] = df["reposts"].apply(lambda x: x.get("count")
    if isinstance(x, dict) else 0)
if "image" in df.columns:
    df["image_url"] = df["image"].apply(lambda x: x[0].get("url"))

cols_to_keep = [
    "id", "owner_id", "title", "description", "duration", "date",
    "views", "comments", "likes_count", "reposts_count", "player", "can_like",
    "can_repost", "can_dislike", "is_pinned", "image_url",
]
existing_cols = [col for col in cols_to_keep if col in df.columns]
df = df[existing_cols]

df.to_csv("videos_2.2.csv", index=False, encoding="utf-8")
print(df.shape)

Видео из сообщества -211229778...
Найдено видео: 79
Видео из сообщества -209976560...
Найдено видео: 349
Видео из сообщества -203677279...
Найдено видео: 585
Видео из сообщества -165221845...
Найдено видео: 118
Видео из сообщества -110135406...
Найдено видео: 1494
Видео из сообщества -159848117...
Найдено видео: 158
Видео из сообщества -149430811...
Найдено видео: 245
(3028, 16)
